# Nerfstudio auf eigenen Bildern — Colab

Linearer Workflow: **Setup → Bilder hochladen → COLMAP-Posen → splatfacto trainieren → Viewer**.

## Bedienung

- **Zelle für Zelle** ausführen (nicht *Run all*).
- Nach Zelle 2 (`condacolab.install_from_url(...)`) **startet die Runtime automatisch neu**. Das ist gewollt. Danach mit Zelle 3 weitermachen — Zelle 2 NICHT erneut.
- Laufzeittyp: **GPU** (Menü → Laufzeit → Laufzeittyp ändern → T4/L4/A100).

Wenn eine Zelle rot wird: stoppen, Output melden. Folgefehler maskieren sonst die Ursache.

## 1. GPU prüfen

In [ ]:
!nvidia-smi

## 2. condacolab + Python 3.10 installieren

Wir verwenden direkt einen Miniconda-Installer, der Python 3.10 als base-env mitbringt — genau die Version, für die das tiny-cuda-nn-Wheel in Zelle 5 gebaut ist.

**Die Runtime restartet automatisch.** Das ist normal — nach dem Restart bei Zelle 3 weitermachen, **nicht erneut Zelle 2**.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url(
    'https://repo.anaconda.com/miniconda/Miniconda3-py310_26.3.2-2-Linux-x86_64.sh'
)

## 3. Conda-Setup verifizieren

Erwartete Ausgabe: `Python 3.10.x` und `condacolab.check()` ohne Fehler.

In [ ]:
import condacolab
condacolab.check()
!python --version
!which python

## 4. CUDA 11.8 + PyTorch 2.0.1

Erwartung am Ende: `torch 2.0.1+cu118 cuda True`.

Wenn `cuda False`: Laufzeittyp auf GPU stellen und Notebook neu starten. Dauer dieser Zelle: ~3–5 Min.

In [ ]:
%%bash
set -e
pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 \
  --extra-index-url https://download.pytorch.org/whl/cu118
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 5. tiny-cuda-nn (vorgebautes Wheel)

Erwartung: `tcnn ok`.

Wenn der gdown-Download stoppt: Drive-Link blockiert dich (manchmal Quota). Dann Zelle erneut ausführen oder Output melden — wir besorgen das Wheel anders.

In [ ]:
%%bash
set -e
pip install -q gdown
WHL=/tmp/tinycudann-1.7-cp310-cp310-linux_x86_64.whl
gdown "https://drive.google.com/u/1/uc?id=1-7x7qQfB7bIw2zV4Lr6-yhvMpjXC84Q5&confirm=t" -O "$WHL"
pip install "$WHL"
python -c "import tinycudann as tcnn; print('tcnn ok')"

## 6. Nerfstudio + COLMAP

Nerfstudio gepinnt auf `1.0.3` (letzte Version mit torch 2.0.1). Erwartung: Liste der `ns-train`-Methoden inkl. `splatfacto`.

In [ ]:
%%bash
set -e
apt-get install -y -qq colmap ffmpeg
pip install nerfstudio==1.0.3
pip install gsplat==0.1.11  # nerfstudio 1.0.3 braucht die alte gsplat._torch_impl API
ns-train --help 2>&1 | head -25

## 7. Bilder aus Google Drive laden

20–80 überlappende Fotos einer Szene. Gleiche Belichtung, keine bewegten Objekte. Format: JPG/PNG.

Bilder liegen in Drive unter **`Meine Ablage / 3DSeminar Bilder / Images`** und werden nach `/content/data/images/` kopiert. Beim ersten Mount erscheint ein Auth-Popup von Google.

In [ ]:
import os, shutil, glob
from google.colab import drive

drive.mount('/content/drive')

SRC_DIR = '/content/drive/MyDrive/3DSeminar Bilder/Images'
IMG_DIR = '/content/data/images'

assert os.path.isdir(SRC_DIR), f'Drive-Ordner nicht gefunden: {SRC_DIR}'
os.makedirs(IMG_DIR, exist_ok=True)

exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
files_list = sorted(f for ext in exts for f in glob.glob(os.path.join(SRC_DIR, ext)))
for src in files_list:
    shutil.copy2(src, os.path.join(IMG_DIR, os.path.basename(src)))

print(f'Kopiert: {len(files_list)} Bilder von {SRC_DIR} nach {IMG_DIR}')

## 8. COLMAP-Posen berechnen

Dauer: ~5–15 Min bei 20–80 Bildern. Ergebnis liegt in `/content/data/processed/` mit `transforms.json` als Index.

In [ ]:
!ns-process-data images \
  --data /content/data/images \
  --output-dir /content/data/processed \
  --matching-method exhaustive

## 9. ngrok-Tunnel für den Viewer (optional, aber empfohlen)

Der Nerfstudio-Viewer läuft auf Port 7007 *in* der Colab-VM — von außen nicht erreichbar. ngrok macht ihn öffentlich.

1. Account bei https://dashboard.ngrok.com/signup (gratis)
2. Token unter https://dashboard.ngrok.com/get-started/your-authtoken kopieren
3. Hier einsetzen, Zelle ausführen — die URL erscheint, und du öffnest sie **nach** dem Trainingsstart in Zelle 10.

In [ ]:
NGROK_TOKEN = '3Dwe7lpwVeba4lxOAxmMD5oxr2k_49JrZmYLQ2sVhGNoEGoPk'

import sys, subprocess, site, importlib
print('Kernel-Python:', sys.executable)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyngrok'])
importlib.reload(site)  # neue site-packages in sys.path aufnehmen

from pyngrok import ngrok, conf
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
tunnel = ngrok.connect(7007, 'http')
print('Viewer-URL (sobald Training läuft):', tunnel.public_url)

## 10. Training starten — splatfacto (3D Gaussian Splatting)

`splatfacto` ist die 3DGS-Variante in Nerfstudio (schnell, gute Qualität, viewer-friendly).

Standard: 30000 Iterationen, ~30–60 Min auf einer T4. Für einen schnellen Test runter auf 7000 setzen (`--max-num-iterations 7000`).

In [ ]:
!ns-train splatfacto \
  --data /content/data/processed \
  --output-dir /content/outputs \
  --viewer.websocket-port 7007 \
  --max-num-iterations 30000

## 13. Komplette Resultate (Checkpoints + Configs) herunterladen

Falls du nicht nur die `.ply` aus Zelle 12 willst, sondern den **kompletten Trainings-Output** (Checkpoints, Configs, Tensorboard-Logs) für späteres Resume/Re-Export. Zelle zippt `/content/outputs/` und lädt es runter.

In [ ]:
import glob
configs = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))
assert configs, 'Kein config.yml gefunden — Training erst durchlaufen lassen.'
CONFIG = configs[-1]
print('Lade Config:', CONFIG)

!ns-viewer --load-config "{CONFIG}" --viewer.websocket-port 7007

## 12. Szene als `.ply` exportieren (für externe Viewer)

Erzeugt `/content/exports/splat.ply` und lädt es runter. Offline anschauen z. B. mit:

- **https://superspl.at/editor** — `.ply` per Drag&Drop in den Browser, fertig.
- PlayCanvas, Polycam, oder jedem 3DGS-fähigen Viewer.

In [ ]:
import glob, os
configs = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))
assert configs, 'Kein config.yml gefunden — Training erst durchlaufen lassen.'
CONFIG = configs[-1]
print('Lade Config:', CONFIG)
os.makedirs('/content/exports', exist_ok=True)

!ns-export gaussian-splat --load-config "{CONFIG}" --output-dir /content/exports
!ls -lh /content/exports/

from google.colab import files
files.download('/content/exports/splat.ply')

## 11. Resultate herunterladen

Trainierte Szene + Checkpoints liegen unter `/content/outputs/`. Zelle zippt alles und lädt es runter.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/outputs', 'zip', '/content/outputs')
files.download('/content/outputs.zip')